# Kapitel 13

In [1]:
BeispielDaten=[[9,2,3],[9,9,9],[7,0,5],[8,4,4],
              [9,3,2],[7,9,8],[5,1,1],[7,2,3]]

In [2]:
import random
def abstand(p:list,q:list)->float: 
   return sum([(p[i]-q[i])**2 for i in range(len(p))])
def MP(daten:list,fallback=[])->list:
    if len(daten)==0: # Leere Daten haben kein Zentrum, wähle zufällig
        daten=[fallback]
    dim=len(daten[0]) # Dimension der einzelnen Vektoren     
    ergebnis=[0]*dim  # Null-Vektor der Dimension dim
    for i in range(dim): # für jede Dimension werden         
        for punkt in daten: # die Eintraege aller Daten              
            ergebnis[i]+=punkt[i]  # addiert     
    return [x/len(daten) for x in ergebnis]
MP([[1,100],[3,200],[8,120]])

[4.0, 140.0]

In [3]:
def kCluster(daten:list,k:int):
    n=len(daten)
    clusterVon=[i%k for i in range(n)] # Willkürliche Startzuordnung
    def PktvClus(j:int)->list: # j ist Cluster-Nummer
        return [daten[i] for i in range(n) if clusterVon[i]==j] 
    geändert=True     
    while geändert:
        geändert=False
        zentren=[MP(PktvClus(j),fallback=random.choice(daten)) for j in range(k)]         
        for i in range(n): # Punkte den Zentren zuordnen
            punkt=daten[i]             
            optimal=0 # Nummer des besten Zentrums 
            for c in range(1,k):
                if (abstand(zentren[c],punkt)<abstand(zentren[optimal],punkt) ):
                    optimal=c        
            if not(clusterVon[i]==optimal):
                geändert=True             
                clusterVon[i]=optimal     
    return [PktvClus(j) for j in range(k)] 
kCluster(BeispielDaten,3)

[[[9, 2, 3], [7, 0, 5], [9, 3, 2], [5, 1, 1], [7, 2, 3]],
 [[8, 4, 4]],
 [[9, 9, 9], [7, 9, 8]]]

In [4]:
def WCSS(clusters:list)->float:
    fehlersumme=0.0     
    for c in clusters:
        m=MP(c)
        for p in c: 
            fehlersumme+=abstand(p,m)
    return fehlersumme
for k in range(1,5):
    print(k," Cluster: WCSS=",WCSS(kCluster(BeispielDaten,k)))

1  Cluster: WCSS= 153.25
2  Cluster: WCSS= 34.0
3  Cluster: WCSS= 27.7
4  Cluster: WCSS= 17.0


In [5]:
def BCSS(daten:list,clusters:list)->float:
    bcss=0.0     
    for c in clusters:
        bcss+=len(c)*abstand(MP(c),MP(daten))
    return bcss
def R2(daten:list,clusters:list)->float:
    bcss=BCSS(daten,clusters)
    wcss=WCSS(clusters)
    return bcss/(wcss+bcss)
for k in range(2,5):
    print(k," Cluster: R2=",R2(BeispielDaten,kCluster(BeispielDaten,k)))

2  Cluster: R2= 0.7781402936378466
3  Cluster: R2= 0.8192495921696575
4  Cluster: R2= 0.8890701468189234


In [6]:
def istVektor(L:list)->bool:
    if not(type(L)==list):
        return False
    for x in L:
        if not(type(x)==int or type(x)==float):
            return False
    return True
def flach(L:list)->list:
    if istVektor(L): 
        return [L]      
    Lflach=[flach(u) for u in L]     
    ergebnis=[]     
    for liste in Lflach: 
        ergebnis=ergebnis+liste     
    return ergebnis  
def SP(daten): 
    return MP(flach(daten))
print(istVektor([4,3,6]), istVektor([[4,3,6]]),istVektor([[],3,6]))
print(flach([[[1,2],[7,8]],[[9,1]],[5,2]]))

True False False
[[1, 2], [7, 8], [9, 1], [5, 2]]


In [7]:
def hcluster(daten):
    C=[p for p in daten] # Eine Kopie
    while len(C)>1: 
        optPaar=[0,1] # Suche vorbereiten         
        optAbstand=abstand(SP(C[0]),SP(C[1]))         
        for i in range(len(C)-1):             
            for j in range(i+1,len(C)):
               if abstand(SP(C[i]),SP(C[j]))<optAbstand:
                   optPaar=[i,j]
                   optAbstand=abstand(SP(C[i]),SP(C[j]))         
        [r,s]=optPaar
        C=C[:r]+C[r+1:s]+C[s+1:]+[[C[r],C[s]]]
    return C[0]
hcluster(BeispielDaten)

[[[9, 9, 9], [7, 9, 8]],
 [[5, 1, 1], [[7, 0, 5], [[8, 4, 4], [[7, 2, 3], [[9, 2, 3], [9, 3, 2]]]]]]]

In [8]:
def printCluster(C,tiefe=0):
    if istVektor(C):
        print( (" "*tiefe)+str(C))
        return
    print((" "*(tiefe-1))+"[")
    for c in C:
        printCluster(c,tiefe+4)
    print((" "*(tiefe-1))+"]")
printCluster(hcluster(BeispielDaten))

[
   [
        [9, 9, 9]
        [7, 9, 8]
   ]
   [
        [5, 1, 1]
       [
            [7, 0, 5]
           [
                [8, 4, 4]
               [
                    [7, 2, 3]
                   [
                        [9, 2, 3]
                        [9, 3, 2]
                   ]
               ]
           ]
       ]
   ]
]
